# Ragionare e agire: il ciclo dell'agente

Il codice del capitolo [«Ragionare e agire: il ciclo dell'agente»](https://book.paithon.it/main/Agenti/agenti-e-tool-use.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q torch torchvision

## Ragionare e agire: il ciclo dell'agente

[Leggi la pagina](https://book.paithon.it/main/Agenti/agenti-e-tool-use.html)


### Un agente giocattolo, in Python


In [ ]:
import astimport operator# --- due strumenti reali ---# operatori ammessi: un mini-interprete sicuro, niente eval()_OP = {    ast.Add: operator.add, ast.Sub: operator.sub,    ast.Mult: operator.mul, ast.Div: operator.truediv,    ast.USub: operator.neg,}def _valuta(nodo):    if isinstance(nodo, ast.Constant):        # un numero        return nodo.value    if isinstance(nodo, ast.BinOp):           # a operatore b        return _OP[type(nodo.op)](_valuta(nodo.left), _valuta(nodo.right))    if isinstance(nodo, ast.UnaryOp):         # -a        return _OP[type(nodo.op)](_valuta(nodo.operand))    raise ValueError("espressione non ammessa")def calcola(espressione):    """Valuta un'espressione aritmetica in modo sicuro (senza eval)."""    return _valuta(ast.parse(espressione, mode="eval").body)# un piccolo archivio: la memoria esterna che il modello non ha nei pesiARCHIVIO = {    "attention is all you need": "2017",    "gpt-3": "2020",    "react": "2023",}def cerca(chiave):    """Cerca un fatto nell'archivio; restituisce sempre una stringa."""    return ARCHIVIO.get(chiave.lower().strip(), "non trovato")STRUMENTI = {"calcola": calcola, "cerca": cerca}

### un piccolo archivio: la memoria esterna che il modello non ha nei pesi


In [ ]:
# --- l'LLM finto: deterministico, a regole ---def llm_finto(traccia):    """Data la traccia finora, emette (pensiero, azione, argomento).    Un vero LLM genererebbe questo testo; qui lo decide una regola."""    ultima = traccia[-1]["osservazione"] if traccia else None    if ultima is None:        return ("Non conosco a memoria l'anno del paper: lo cerco.",                "cerca", "attention is all you need")    if ultima == "2017":        return ("Il paper è del 2017. Calcolo quanti anni fa, dal 2026.",                "calcola", "2026 - 2017")    return (f"Il calcolo dice {ultima}: ho tutto per rispondere.",            "Answer", "'Attention Is All You Need' è del 2017: 9 anni fa nel 2026.")# --- il ciclo dell'agente ---def esegui_agente(domanda, max_passi=5):    print(f"Domanda: {domanda}\n")    traccia = []    for _ in range(max_passi):        pensiero, azione, argomento = llm_finto(traccia)   # il modello "ragiona"        print(f"Thought: {pensiero}")        if azione == "Answer":                             # fine del loop            print(f"Answer: {argomento}")            return argomento        print(f"Action: {azione}[{argomento}]")        osservazione = str(STRUMENTI[azione](argomento))   # il sistema agisce        print(f"Observation: {osservazione}\n")        traccia.append({"azione": azione, "argomento": argomento,                        "osservazione": osservazione})      # torna nel contesto    print("(limite di passi raggiunto)")esegui_agente("In che anno è uscito 'Attention Is All You Need' "              "e quanti anni fa è, nel 2026?")

## RAG avanzato: oltre il recupero ingenuo

[Leggi la pagina](https://book.paithon.it/main/Agenti/rag-avanzato.html)


### Riordinare i candidati: il reranking


In [ ]:
import torch# --- Stadio 1: recupero grezzo con il bi-encoder ---# stesso mini-archivio della sezione «Cercare per rispondere»:# quattro dimensioni leggibili [gatti, muri/casa, automobili, cucina].passaggi = [    "Il gatto nero salta sul muro del giardino.",    "Il muro portante sostiene il solaio.",    "La vettura elettrica si ricarica in garage.",    "L'auto storica sfila per il centro.",    "Il gatto dorme accanto ai fornelli.",    "La ricetta prevede burro e salvia.",]E = torch.tensor([    [0.9, 0.6, 0.0, 0.1],    [0.1, 0.9, 0.1, 0.0],    [0.0, 0.1, 0.9, 0.0],    [0.1, 0.0, 0.8, 0.1],    [0.8, 0.2, 0.0, 0.5],    [0.0, 0.1, 0.1, 0.9],])E = E / E.norm(dim=1, keepdim=True)   # righe normalizzate: prodotto = cosenodomanda = "Su cosa salta il gatto nero?"q = torch.tensor([0.9, 0.7, 0.0, 0.0])q = q / q.norm()# il bi-encoder e' veloce: puo' spazzare tutto l'archivio, ma e' grezzo.# recuperiamo piu' candidati del necessario (over-retrieval): sono il# terreno di caccia dello stadio successivo.sim = E @ qval, cand = torch.topk(sim, k=4)print("Stadio 1 - bi-encoder (veloce, tutto l'archivio):")for v, i in zip(val, cand):    print(f"  coseno {v:.2f}  {passaggi[i]}")# --- Stadio 2: reranking con un "cross-encoder" didattico ---# Il bi-encoder ha collassato ogni frase in un unico vettore: cosi' un# passaggio che condivide solo il tema "gatto" gli sembra vicino. Un# cross-encoder legge domanda e passaggio INSIEME e distingue il tema# dall'azione. Qui lo simuliamo con dei concetti pesati: il tema (gatto)# conta poco, l'azione (saltare) e il bersaglio (muro) molto.concetti_domanda = {"gatto": 1.0, "saltare": 3.0, "muro": 3.0}concetti_passaggio = {    0: {"gatto", "saltare", "muro"},      # gatto che SALTA sul MURO: risponde    1: {"muro", "solaio"},    2: {"vettura", "garage"},    3: {"auto", "centro"},    4: {"gatto", "dormire", "fornelli"},  # solo il tema "gatto": quasi-pertinente    5: {"ricetta", "burro"},}def cross_encoder(i):    # pertinenza = peso dei concetti della domanda davvero coperti dal passaggio    coperti = concetti_passaggio[i]    return sum(peso for c, peso in concetti_domanda.items() if c in coperti)# il cross-encoder e' costoso: lo applichiamo SOLO ai candidati dello stadio 1riordino = sorted(cand.tolist(), key=cross_encoder, reverse=True)print("\nStadio 2 - cross-encoder (preciso, solo sui 4 candidati):")for i in riordino:    print(f"  pertinenza {cross_encoder(i):.1f}  {passaggi[i]}")print("\nAl generatore vanno i primi due dopo il reranking:")for n, i in enumerate(riordino[:2], 1):    print(f"  [{n}] {passaggi[i]}")

## Context engineering: il contesto è l'interfaccia

[Leggi la pagina](https://book.paithon.it/main/Agenti/context-engineering.html)


### Assemblare il contesto, con un budget


In [ ]:
# Un "context builder": dato un budget di token, assembla il prompt# scegliendo i passaggi piu' importanti, troncando o scartando il resto,# e collocando il pezzo piu' rilevante IN FONDO (contro il "lost in the middle").def conta_token(testo):    """Stima i token contando le parole: grezza, ma sufficiente per il budget."""    return len(testo.split())# System prompt e domanda sono obbligatori: entrano sempre, non si toccano.system_prompt = (    "Sei un assistente che risponde citando solo i passaggi forniti. "    "Se l'informazione non c'e', dillo.")domanda = "In che anno e' stato pubblicato il paper sui Transformer?"# I passaggi recuperati, ciascuno con una rilevanza (piu' alta = piu' utile).passaggi = [    (0.95, "Il paper 'Attention Is All You Need' introduce i Transformer nel 2017."),    (0.20, "Le reti convoluzionali dominarono la visione artificiale negli anni 2010."),    (0.60, "L'architettura Transformer abbandona la ricorrenza in favore dell'attenzione."),    (0.10, "Il primo modello GPT fu addestrato su un corpus di libri."),    (0.75, "L'attenzione scaled dot-product e' il cuore del Transformer."),]def costruisci_contesto(system_prompt, passaggi, domanda, budget):    """Assembla un prompt che sta nel budget di token.    Obbligatori: system prompt e domanda. I passaggi entrano per rilevanza    decrescente finche' c'e' spazio; l'ultimo che sfora viene troncato; il    piu' rilevante finisce in fondo, appena sopra la domanda."""    residuo = budget - conta_token(system_prompt) - conta_token(domanda)    if residuo < 0:        raise ValueError("budget insufficiente perfino per system prompt e domanda")    ordinati = sorted(passaggi, key=lambda p: p[0], reverse=True)    scelti = []  # (punteggio, testo, troncato?)    for punteggio, testo in ordinati:        costo = conta_token(testo)        if costo <= residuo:                       # ci sta intero            scelti.append((punteggio, testo, False))            residuo -= costo        elif residuo >= 4:                         # non ci sta: lo tronco per riempire            troncato = " ".join(testo.split()[:residuo - 1]) + " …"            scelti.append((punteggio, troncato, True))            residuo -= conta_token(troncato)            break        # altrimenti lo scarto e provo il prossimo (piu' corto o meno rilevante)    # "lost in the middle": rilevanza crescente, cosi' il migliore va per ultimo.    scelti.sort(key=lambda p: p[0])    righe = [f"[fonte {p:.2f}{' (troncata)' if t else ''}] {txt}"             for p, txt, t in scelti]    corpo = "\n".join(righe)    prompt = f"{system_prompt}\n\n{corpo}\n\nDomanda: {domanda}"    return prompt, budget - residuoprompt, usati = costruisci_contesto(system_prompt, passaggi, domanda, budget=50)print(prompt)print(f"\nToken usati: {usati}/50")

## Architetture di agenti e come valutarli

[Leggi la pagina](https://book.paithon.it/main/Agenti/architetture-e-valutazione.html)


### Valutare un agente: il problema difficile


In [ ]:
# ogni episodio: esito, passi, token consumati, traiettoria valida?episodi = [    {"successo": True,  "passi": 4,  "token": 2100, "traiettoria_ok": True},    {"successo": True,  "passi": 9,  "token": 5400, "traiettoria_ok": False},    {"successo": False, "passi": 12, "token": 8000, "traiettoria_ok": False},    {"successo": True,  "passi": 5,  "token": 2600, "traiettoria_ok": True},    {"successo": False, "passi": 6,  "token": 3100, "traiettoria_ok": True},]n = len(episodi)successi = [e for e in episodi if e["successo"]]tasso_successo = len(successi) / n# fra i compiti riusciti, quanti per una strada "pulita"?traiettorie_ok = sum(e["traiettoria_ok"] for e in successi) / len(successi)token_medi = sum(e["token"] for e in episodi) / nprint(f"episodi: {n}")print(f"tasso di successo: {tasso_successo:.0%}")print(f"successi con traiettoria valida: {traiettorie_ok:.0%}")print(f"token medi per episodio: {token_medi:.0f}")